In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores, CAL_select
from loka.geometrical_methods import herding_order, k_center_order

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

BASE = os.getenv("BASE_DIR")

X,Y = load_breakthrough_states(f"{BASE}/data/game_states/e.txt")
print(X.shape)

In [ ]:
h_order = herding_order(X, 0.11, device)

In [ ]:
kc_order = k_center_order(X, 0.11, device)

In [ ]:
orders = [h_order, kc_order]

In [ ]:
def train(model, optimizer, loss_fn, X, Y, epochs, pbar, pbar_inc, n_points):
    X_t = X[:n_points].to(device)
    Y_t = Y[:n_points].to(device)

    for _ in range(epochs):
        perm = torch.randperm(len(X_t), device=device)
        
        for i in range(0, len(X_t), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X_t[idx], Y_t[idx]

            optimizer.zero_grad()
            batch_loss = loss_fn(model(xb), yb)
            batch_loss.backward()
            optimizer.step()
        
        pbar.update(pbar_inc)

In [ ]:
TOTAL_EPOCHS = 50
BATCH_SIZE = 512
AL_ROUNDS = 20
INIT_X_LABEL_SIZE = 0.1
K_NEAREST = 20  

start, end, steps = 0.01, 0.1, 50
splits = np.linspace(start, end, num=steps, endpoint=True)

n_points_per_split = [int(s * len(X)) for s in splits]
total_work = len(orders) * TOTAL_EPOCHS * sum(
    sum(int(S * INIT_X_LABEL_SIZE) + r * ((S - int(S * INIT_X_LABEL_SIZE)) // AL_ROUNDS)
        for r in range(AL_ROUNDS + 1))
    for S in n_points_per_split
)

all_mse_results = list()
all_prc_results = list()

pbar = tqdm(total=total_work, desc='Training',
            bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]")

X_t = torch.FloatTensor(X).to(device)      # once, outside all loops
Y_t = torch.FloatTensor(Y).to(device)         

for order in orders:
    mse_values, prc_values = list(), list()

    for s, pbar_inc in zip(splits, n_points_per_split):
        n_points = int(s * len(X))
        S_size = n_points                                        # typo fixed (S_Size)
        init_x_label_size = int(S_size * INIT_X_LABEL_SIZE)
        size_Q = (S_size - init_x_label_size) // AL_ROUNDS

        init_idx = order[:init_x_label_size]
        X_label, Y_label = X_t[init_idx], Y_t[init_idx]

        mask = torch.ones(len(X_t), dtype=torch.bool, device=device)
        mask[order[:S_size]] = True                        
        mask[:] = False
        mask[order[:S_size]] = True
        mask[init_idx] = False                          
        X_pool, Y_pool = X_t[mask], Y_t[mask]

        torch.manual_seed(SEED)
        model = BreakthroughCNN().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()

        for _ in range(AL_ROUNDS):
            # train first, then pick
            train(model, optimizer, loss_fn, X_label, Y_label, TOTAL_EPOCHS, pbar, len(X_label), len(X_label))

            Q_idx = CAL_select(model, X_label, X_pool, size_Q, k_nearest=K_NEAREST)

            X_label = torch.cat([X_label, X_pool[Q_idx]], dim=0)
            Y_label = torch.cat([Y_label, Y_pool[Q_idx]], dim=0)   # oracle step

            keep = torch.ones(len(X_pool), dtype=torch.bool, device=device)
            keep[Q_idx] = False
            X_pool, Y_pool = X_pool[keep], Y_pool[keep]

        # final train on the full labeled set, then evaluate
        train(model, optimizer, loss_fn, X_label, Y_label, TOTAL_EPOCHS, pbar, len(X_label), len(X_label))

        model.eval()
        with torch.no_grad():
            sq_err = 0.0
            for k in range(0, len(X_t), BATCH_SIZE):
                xb, yb = X_t[k:k + BATCH_SIZE], Y_t[k:k + BATCH_SIZE]
                sq_err += ((model(xb) - yb) ** 2).sum().item()
            mse = sq_err / Y_t.numel()
        mse_values.append(mse)
        prc_values.append(s)

        del model, optimizer
        torch.cuda.empty_cache()

    all_mse_results.append(mse_values)
    all_prc_results.append(prc_values)

pbar.close()